# Extract Scalar Performance of models from a log file

## Fichier E13sg_9_a2c_run_all_agents_test.log

In [ ]:
import re
import pandas as pd

# Chemin vers votre fichier de log
log_file_path = 'results/E13sg_run_all_agents_test.log'

# Listes pour stocker les données extraites
data = []

# Expressions régulières
pattern_result = re.compile(
    r'Trial:\s*(\d+).*?TestPerf\(ScalarPerf,USLA,RAM,SLAV\):\s*\(([^,]+),\s*([^,]+),\s*([^,]+),\s*([^\)]+)\)'
)

pattern_agent = re.compile(
    r'AgentFilename:\s*\.\/outputs\/([^\/]+)\/agent-([^_]+)-([^_]+)_'
)

# Variable pour stocker temporairement les infos Agent
current_agent_info = None

# Fonction pour extraire "version" et "simu" à partir du nom de dossier
def extract_version_simu(folder_name):
    # Exemple : E13sg_9_tabddpm-a2c-training_disq
    parts = folder_name.split('_')
    if len(parts) >= 3:
        version = parts[0]
        version_minor = parts[1]  # E13sg_9
        simu = parts[2].split('-')[0]     # tabddpm
        return version, version_minor, simu
    return None, None, None

# Lecture du fichier de log
with open(log_file_path, 'r') as file:
    for line in file:
        # Vérifier si la ligne est une ligne AgentFilename
        match_agent = pattern_agent.search(line)
        if match_agent:
            folder_name = match_agent.group(1)
            version, version_minor, simu = extract_version_simu(folder_name)
            model = match_agent.group(3)
            # Stocker temporairement ces infos
            current_agent_info = {
                'Version': version,
                'VersionMinor': version_minor,
                'Simu': simu,
                'Model': model
            }
        # Vérifier si la ligne est une ligne TestPerf
        match_result = pattern_result.search(line)
        if match_result:
            trial_num = int(match_result.group(1))
            scalar_perf = float(match_result.group(2))
            usla = int(match_result.group(3))
            ram = int(match_result.group(4))
            slav = int(match_result.group(5))
            # Créer l'entrée avec les résultats
            entry = {
                'Trial': trial_num,
                'ScalarPerf': scalar_perf,
                'USLA': usla,
                'RAM': ram,
                'SLAV': slav,
                'Version': None,
                'VersionMinor': None,
                'Simu': None,
                'Model': None
            }
            # Ajouter les infos Agent si disponibles
            if current_agent_info:
                entry.update(current_agent_info)
            data.append(entry)

# Convertir en DataFrame
df = pd.DataFrame(data)
# Afficher le DataFrame final
#print(df)
version_list = df['Version'].unique()
version_minor_list = df['VersionMinor'].unique()
simu_list = df['Simu'].unique()
model_list = df['Model'].unique()
print("Versions found in the log file:")
print(version_list)
print(version_minor_list)
print(simu_list)
print(model_list)
roundat=4

# Generate result table
results = []
for version in version_list:
    for version_minor in version_minor_list:
        for simu in simu_list:
            for model in model_list:
                dfperf = df.loc[(df['Version'] == version) & (df['VersionMinor'] == version_minor) & (df['Simu'] == simu) & (df['Model'] == model)]
                if not dfperf.empty:
                    dfperf = df.loc[(df['Version'] == version) & (df['VersionMinor'] == version_minor) & (df['Simu'] == simu) & (df['Model'] == model)]
                    scalar_perf_min = dfperf['ScalarPerf'].min()
                    scalar_perf_max = dfperf['ScalarPerf'].max()
                    dfmin = dfperf.loc[dfperf['ScalarPerf'] == scalar_perf_min]
                    dfmax = dfperf.loc[dfperf['ScalarPerf'] == scalar_perf_max]

                    scalar_perf_max = round(dfperf['ScalarPerf'].max(), roundat)
                    scalar_perf_min = round(scalar_perf_min, roundat)
                    scalar_perf_std = round(dfperf['ScalarPerf'].std(), roundat)
                    #print(f"{version}_{version_minor} {simu} {model}")
                    #perfmin = f'({scalar_perf_min}, {dfmin.iloc[0]["USLA"]}, {dfmin.iloc[0]["RAM"]}, {dfmin.iloc[0]["SLAV"]})'
                    #perfmax = f'({scalar_perf_max}, {dfmax.iloc[0]["USLA"]}, {dfmax.iloc[0]["RAM"]}, {dfmax.iloc[0]["SLAV"]})'
                    #print(f"{version}_{version_minor} {simu} {model} Min-Max(ScalarPerf,USLA,RAM,SLAV): {perfmin}-{perfmax} NormStdDev: {scalar_perf_std}")
                    entry = {
                        'Version': version+"_"+version_minor,
                        'Simu': str(simu),
                        'Model': model,
                        'PerfMin': scalar_perf_min,
                        'PerfMax': scalar_perf_max,
                        'PerfStd': scalar_perf_std,
                        'USLAMin': dfmin.iloc[0]["USLA"],
                        'USLAMax': dfmax.iloc[0]["USLA"],
                        'RAMMin': dfmin.iloc[0]["RAM"],
                        'RAMMax': dfmax.iloc[0]["RAM"],
                        'SLAVMin': dfmin.iloc[0]["SLAV"],
                        'SLAVMax': dfmax.iloc[0]["SLAV"],
                    }
                    results.append(entry)

# Convertir en DataFrame
dfresults = pd.DataFrame(results)

# Afficher le DataFrame final
pd.options.display.width = 1000
print(dfresults)


     Trial  ScalarPerf  USLA       RAM  SLAV Version VersionMinor     Simu   Model
0        0    0.354637   335  28768768     0   E13sg            8     orig  SB3A2C
1        1    0.356120   335  28800256     0   E13sg            8     orig  SB3A2C
2        2    0.369181   335  29077632     0   E13sg            8     orig  SB3A2C
3        3    0.342860   335  28518656     0   E13sg            8     orig  SB3A2C
4        4    0.363051   335  28947456     0   E13sg            8     orig  SB3A2C
..     ...         ...   ...       ...   ...     ...          ...      ...     ...
125      5    0.910769   354  39374976    14   E13sg            9  tabddpm  SB3DQN
126      6    1.020118   353  41760640    11   E13sg            9  tabddpm  SB3DQN
127      7    1.111915   355  43583360    15   E13sg            9  tabddpm  SB3DQN
128      8    1.038404   360  41705216    17   E13sg            9  tabddpm  SB3DQN
129      9    1.043782   354  42199808    15   E13sg            9  tabddpm  SB3DQN

[13